# 基于傅立叶相位差推断交通图边方向（多日拼接版）

- 将多天数据拼接成长序列，提高频率分辨率
- 使用FFT计算相位差来估计时间延迟
- 相比单日版本，相位估计更稳定

## 1. 环境配置

In [ ]:
import pandas as pd
import numpy as np
from collections import Counter
import matplotlib.pyplot as plt
import os
import warnings
warnings.filterwarnings('ignore')

OUTPUT_DIR = "../output/edge_inference"
META_DIR = "../d03_meta_processed"
DATA_DIR = "/data/yuzhang_fei/PEMS/PEMSD3_2025"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("库加载完成！")

## 2. 加载数据

In [ ]:
# 加载交通流数据
print("加载交通流数据...")
flow_df = pd.read_csv(
    os.path.join(DATA_DIR, 'pems_5min_merged.csv'),
    dtype={'Station': str},
    parse_dates=['Timestamp']
)
print(f"交通流数据: {len(flow_df):,} 行")

# 加载 ML 传感器元数据
print("\n加载传感器元数据...")
meta_df = pd.read_csv(
    os.path.join(META_DIR, 'sensors_common_2025.csv'),
    dtype={'ID': str, 'Fwy': str}
)
ml_meta = meta_df[meta_df['Type'] == 'ML'].copy()
print(f"ML 传感器数: {len(ml_meta)}")

# 筛选 ML 传感器的流量数据
ml_ids = set(ml_meta['ID'])
flow_ml = flow_df[flow_df['Station'].isin(ml_ids)].copy()
print(f"ML 流量数据: {len(flow_ml):,} 行")

In [ ]:
# 提取日期信息
flow_ml['Date'] = flow_ml['Timestamp'].dt.date
flow_ml['DayOfWeek'] = flow_ml['Timestamp'].dt.dayofweek  # 0=Mon
flow_ml['TimeSlot'] = flow_ml['Timestamp'].dt.hour * 12 + flow_ml['Timestamp'].dt.minute // 5

# 查看可用日期
available_dates = sorted(flow_ml['Date'].unique())
print(f"可用日期数: {len(available_dates)}")
print(f"日期范围: {available_dates[0]} ~ {available_dates[-1]}")

## 3. 选择用于分析的日期

**多日拼接方法需要选择连续的多天数据**

In [ ]:
# 选择连续的多天（建议 7-14 天工作日）
selected_dates = [
    pd.Timestamp('2025-01-06').date(),  # Mon
    pd.Timestamp('2025-01-07').date(),  # Tue
    pd.Timestamp('2025-01-08').date(),  # Wed
    pd.Timestamp('2025-01-09').date(),  # Thu
    pd.Timestamp('2025-01-10').date(),  # Fri
    pd.Timestamp('2025-01-11').date(),  # Mon
    pd.Timestamp('2025-01-12').date(),  # Tue
]

print(f"选择的分析日期 ({len(selected_dates)} 天):")
for d in selected_dates:
    print(f"  {d} ({d.strftime('%A')})")
print(f"\n拼接后序列长度: {len(selected_dates) * 288} 点")

## 4. 构建相邻传感器对

In [ ]:
def build_adjacent_pairs(ml_meta):
    """
    构建相邻传感器对，确定理论方向
    """
    pairs = []
    
    for (fwy, direction), group in ml_meta.groupby(['Fwy', 'Dir']):
        sorted_group = group.sort_values('Abs_PM').reset_index(drop=True)
        
        for i in range(len(sorted_group) - 1):
            node1 = sorted_group.iloc[i]
            node2 = sorted_group.iloc[i + 1]
            
            distance = node2['Abs_PM'] - node1['Abs_PM']
            
            # 理论方向
            if direction in ['N', 'E']:
                source, target = node1['ID'], node2['ID']
            else:
                source, target = node2['ID'], node1['ID']
            
            pairs.append({
                'Node1': node1['ID'],
                'Node2': node2['ID'],
                'Fwy': fwy,
                'Dir': direction,
                'Distance': distance,
                'True_Source': source,
                'True_Target': target
            })
    
    return pd.DataFrame(pairs)


pairs_df = build_adjacent_pairs(ml_meta)
print(f"相邻传感器对数: {len(pairs_df)}")

## 5. 构建数据缓存

In [ ]:
# 构建 pivot 表加速查询
print("构建数据索引...")
flow_pivot = flow_ml.pivot_table(
    index=['Station', 'Date'],
    columns='TimeSlot',
    values='Total_Flow',
    aggfunc='first'
)
print(f"Pivot 表形状: {flow_pivot.shape}")

## 6. 傅立叶相位差计算（多日拼接版）

**核心改进**：
- 单日版本：288 点，1 个完整周期，频率分辨率 = 1/288
- 多日拼接：288×N 点，N 个完整周期，频率分辨率 = 1/(288×N)

**原理**：
- 时移定理：若 B(t) = A(t - Δt)，则 FFT(B) = FFT(A) × e^(-j·2π·f·Δt)
- 相位差 Δφ(f) = φ_A(f) - φ_B(f)
- 时间延迟 Δt = Δφ / (2π) × T

In [ ]:
def get_multiday_series(flow_pivot, station, dates):
    """
    获取多天拼接的时间序列
    
    参数:
    - flow_pivot: pivot 表
    - station: 站点 ID
    - dates: 日期列表
    
    返回:
    - 拼接后的序列（len = 288 × len(dates)）
    """
    series_list = []
    
    for date in dates:
        key = (station, date)
        try:
            if key in flow_pivot.index:
                day_series = flow_pivot.loc[key].values
                series_list.append(day_series)
            else:
                series_list.append(np.full(288, np.nan))
        except:
            series_list.append(np.full(288, np.nan))
    
    return np.concatenate(series_list)

In [ ]:
def compute_lag_fft_multiday(series1, series2, n_days, target_periods_days=[1.0, 0.5, 1/3]):
    """
    使用多日拼接数据计算 FFT 相位差
    
    参数:
    - series1, series2: 多日拼接序列（288 × n_days 点）
    - n_days: 天数
    - target_periods_days: 目标周期（以天为单位）
        - 1.0: 日周期
        - 0.5: 半日周期（12h）
        - 1/3: 8h 周期
    
    返回:
    - lag: 时间延迟（步数），正值表示 series1 领先
    - coherence: 平均相干性
    """
    if series1 is None or series2 is None:
        return np.nan, np.nan
    
    # 去除 NaN
    mask = ~(np.isnan(series1) | np.isnan(series2))
    s1 = series1[mask]
    s2 = series2[mask]
    
    n = len(s1)
    # 至少需要 2 天的有效数据
    if n < 288 * 2:
        return np.nan, np.nan
    
    std1, std2 = np.std(s1), np.std(s2)
    if std1 < 1e-6 or std2 < 1e-6:
        return np.nan, np.nan
    
    # 去均值
    s1 = s1 - np.mean(s1)
    s2 = s2 - np.mean(s2)
    
    # FFT
    fft1 = np.fft.fft(s1)
    fft2 = np.fft.fft(s2)
    freqs = np.fft.fftfreq(n)
    
    # 交叉功率谱
    cross_spectrum = fft1 * np.conj(fft2)
    power1 = np.abs(fft1) ** 2
    power2 = np.abs(fft2) ** 2
    
    phase_diffs = []
    coherences = []
    weights = []
    
    for period_days in target_periods_days:
        # 转换为采样点数
        period_samples = period_days * 288
        
        if period_samples > n:
            continue
        
        # 目标频率
        target_freq = 1.0 / period_samples
        freq_idx = np.argmin(np.abs(freqs[:n//2] - target_freq))
        
        if freq_idx == 0:
            continue
        
        # 相位差
        phase_diff = np.angle(cross_spectrum[freq_idx])
        
        # 相干性
        denom = power1[freq_idx] * power2[freq_idx]
        if denom < 1e-10:
            continue
        coh = np.abs(cross_spectrum[freq_idx]) ** 2 / denom
        coh = np.sqrt(coh)
        
        # 能量权重
        energy = np.abs(cross_spectrum[freq_idx])
        
        # 多日数据，提高相干性阈值
        if coh > 0.5:
            phase_diffs.append(phase_diff)
            coherences.append(coh)
            weights.append(energy * coh)
    
    if len(phase_diffs) == 0:
        return np.nan, np.nan
    
    # 加权平均
    weights = np.array(weights)
    weights = weights / np.sum(weights)
    avg_phase_diff = np.sum(np.array(phase_diffs) * weights)
    avg_coherence = np.sum(np.array(coherences) * weights)
    
    # 相位差转时间延迟（使用日周期 288 点作为参考）
    lag = avg_phase_diff / (2 * np.pi) * 288
    
    # 限制在合理范围（±半个周期 = ±144步 = ±12小时）
    if lag > 144:
        lag -= 288
    elif lag < -144:
        lag += 288
    
    return lag, avg_coherence

## 7. 批量分析（优化版）

In [ ]:
def analyze_all_pairs_fft_multiday(flow_pivot, pairs_df, selected_dates):
    """
    使用多日拼接数据进行 FFT 分析
    """
    n_pairs = len(pairs_df)
    n_days = len(selected_dates)
    
    print(f"使用 {n_days} 天拼接数据 ({n_days * 288} 点/序列)")
    
    # 预提取所有多日拼接序列
    print("提取多日序列到缓存...")
    series_cache = {}
    all_stations = set(pairs_df['Node1'].tolist() + pairs_df['Node2'].tolist())
    
    for station in all_stations:
        series_cache[station] = get_multiday_series(flow_pivot, station, selected_dates)
    
    print(f"缓存序列数: {len(series_cache)}")
    
    # 批量计算
    print("计算 FFT 相位差...")
    results = []
    
    for i, row in pairs_df.iterrows():
        node1, node2 = row['Node1'], row['Node2']
        
        s1 = series_cache.get(node1)
        s2 = series_cache.get(node2)
        
        lag, coh = compute_lag_fft_multiday(s1, s2, n_days)
        
        # 方向判断
        if np.isnan(lag):
            pred_source = 'unknown'
        elif lag > 0.5:
            pred_source = node1
        elif lag < -0.5:
            pred_source = node2
        else:
            pred_source = 'uncertain'
        
        result = {
            'Node1': node1,
            'Node2': node2,
            'Fwy': row['Fwy'],
            'Dir': row['Dir'],
            'Distance': row['Distance'],
            'True_Source': row['True_Source'],
            'Pred_Source': pred_source,
            'Lag': lag,
            'Coherence': coh
        }
        results.append(result)
        
        if (i + 1) % 100 == 0:
            print(f"  {i + 1}/{n_pairs}")
    
    print("分析完成！")
    return pd.DataFrame(results)

In [ ]:
# 配置
METRIC = 'Total_Flow'

print(f"配置:")
print(f"  使用指标: {METRIC}")
print(f"  拼接天数: {len(selected_dates)}")
print(f"  序列长度: {len(selected_dates) * 288} 点")
print(f"  传感器对数: {len(pairs_df)}")

# 运行分析
print(f"\n开始分析...")
results_df = analyze_all_pairs_fft_multiday(flow_pivot, pairs_df, selected_dates)

## 8. 评估精度

In [ ]:
# 判断正确性
results_df['Correct'] = results_df['Pred_Source'] == results_df['True_Source']

# 统计
total = len(results_df)
uncertain = (results_df['Pred_Source'] == 'uncertain').sum()
unknown = (results_df['Pred_Source'] == 'unknown').sum()
valid = results_df[~results_df['Pred_Source'].isin(['uncertain', 'unknown'])]
correct = results_df['Correct'].sum()

print("=" * 60)
print("FFT 相位差方法（多日拼接版）- 精度评估")
print("=" * 60)
print(f"拼接天数: {len(selected_dates)}")
print(f"序列长度: {len(selected_dates) * 288} 点")
print(f"\n总传感器对数: {total}")
print(f"有效预测数: {len(valid)} ({len(valid)/total*100:.1f}%)")
print(f"不确定数: {uncertain} ({uncertain/total*100:.1f}%)")
print(f"无数据数: {unknown} ({unknown/total*100:.1f}%)")
print(f"\n正确预测数: {correct}")
print(f"整体精度: {correct/total*100:.1f}%")
if len(valid) > 0:
    print(f"有效预测精度: {valid['Correct'].sum()/len(valid)*100:.1f}%")

In [ ]:
# 按方向统计
print("\n按方向统计:")
dir_stats = results_df.groupby('Dir').agg({
    'Correct': ['sum', 'count', 'mean']
}).round(3)
dir_stats.columns = ['正确数', '总数', '精度']
display(dir_stats)

In [ ]:
# 按距离统计
print("\n按距离统计:")
results_df['Dist_Bin'] = pd.cut(
    results_df['Distance'],
    bins=[0, 0.5, 1, 2, 5, 100],
    labels=['<0.5mi', '0.5-1mi', '1-2mi', '2-5mi', '>5mi']
)
dist_stats = results_df.groupby('Dist_Bin').agg({
    'Correct': ['sum', 'count', 'mean']
}).round(3)
dist_stats.columns = ['正确数', '总数', '精度']
display(dist_stats)

In [ ]:
# 按相干性统计
print("\n按相干性统计:")
results_df['Coh_Bin'] = pd.cut(
    results_df['Coherence'],
    bins=[0, 0.6, 0.7, 0.8, 0.9, 1.0],
    labels=['0.5-0.6', '0.6-0.7', '0.7-0.8', '0.8-0.9', '0.9-1.0']
)
coh_stats = results_df.groupby('Coh_Bin').agg({
    'Correct': ['sum', 'count', 'mean']
}).round(3)
coh_stats.columns = ['正确数', '总数', '精度']
display(coh_stats)

## 9. 可视化

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. 滞后分布
valid_lags = results_df['Lag'].dropna()
axes[0, 0].hist(valid_lags, bins=50, color='steelblue', edgecolor='white')
axes[0, 0].axvline(x=0, color='red', linestyle='--', linewidth=2)
axes[0, 0].set_xlabel('Lag (steps, 5min each)')
axes[0, 0].set_ylabel('Count')
axes[0, 0].set_title(f'Distribution of Lag (FFT Multi-day, n={len(selected_dates)})')

# 2. 相干性分布
valid_coh = results_df['Coherence'].dropna()
axes[0, 1].hist(valid_coh, bins=30, color='green', edgecolor='white')
axes[0, 1].axvline(x=0.5, color='red', linestyle='--', alpha=0.5, label='Threshold')
axes[0, 1].set_xlabel('Coherence')
axes[0, 1].set_ylabel('Count')
axes[0, 1].set_title('Distribution of Coherence')
axes[0, 1].legend()

# 3. 精度 vs 距离
dist_acc = results_df.groupby('Dist_Bin')['Correct'].mean()
dist_acc.plot(kind='bar', ax=axes[1, 0], color='orange', edgecolor='white')
axes[1, 0].set_xlabel('Distance')
axes[1, 0].set_ylabel('Accuracy')
axes[1, 0].set_title('Accuracy by Distance')
axes[1, 0].set_xticklabels(axes[1, 0].get_xticklabels(), rotation=45)
axes[1, 0].set_ylim(0, 1)

# 4. 滞后 vs 相干性（颜色=正确性）
valid_df = results_df.dropna(subset=['Lag', 'Coherence'])
colors = ['green' if c else 'red' for c in valid_df['Correct']]
axes[1, 1].scatter(valid_df['Lag'], valid_df['Coherence'], c=colors, alpha=0.5, s=20)
axes[1, 1].axvline(x=0, color='gray', linestyle='--', alpha=0.5)
axes[1, 1].axhline(y=0.5, color='gray', linestyle='--', alpha=0.5)
axes[1, 1].set_xlabel('Lag (steps)')
axes[1, 1].set_ylabel('Coherence')
axes[1, 1].set_title('Lag vs Coherence (green=correct, red=wrong)')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'direction_inference_fft_multiday.png'), dpi=150)
plt.show()

In [ ]:
# 查看具体案例
print("\n预测正确的案例:")
display(results_df[results_df['Correct']][['Node1', 'Node2', 'Fwy', 'Dir', 'Distance', 
                                            'Lag', 'Coherence']].head(10))

print("\n预测错误的案例:")
wrong = results_df[~results_df['Correct'] & ~results_df['Pred_Source'].isin(['uncertain', 'unknown'])]
display(wrong[['Node1', 'Node2', 'Fwy', 'Dir', 'Distance', 'Lag', 
               'Coherence', 'True_Source', 'Pred_Source']].head(10))

## 10. 保存结果

In [ ]:
# 保存完整结果
output_path = os.path.join(OUTPUT_DIR, 'direction_inference_fft_multiday.csv')
results_df.to_csv(output_path, index=False)
print(f"结果已保存: {output_path}")

# 保存摘要
summary = f"""
边方向推断结果摘要（FFT相位差 - 多日拼接版）
================================================

配置:
- 使用指标: {METRIC}
- 拼接天数: {len(selected_dates)}
- 序列长度: {len(selected_dates) * 288} 点
- 目标周期: 24h, 12h, 8h
- 相干性阈值: 0.5

分析日期:
{chr(10).join(str(d) for d in selected_dates)}

结果:
- 总传感器对数: {total}
- 有效预测数: {len(valid)} ({len(valid)/total*100:.1f}%)
- 不确定数: {uncertain} ({uncertain/total*100:.1f}%)
- 无数据数: {unknown} ({unknown/total*100:.1f}%)

精度:
- 整体精度: {correct/total*100:.1f}%
- 有效预测精度: {valid['Correct'].sum()/len(valid)*100:.1f}%

按方向:
{dir_stats.to_string()}

按距离:
{dist_stats.to_string()}
"""

with open(os.path.join(OUTPUT_DIR, 'direction_inference_fft_multiday_summary.txt'), 'w') as f:
    f.write(summary)
print(f"摘要已保存: direction_inference_fft_multiday_summary.txt")

---
## 说明

### 多日拼接 vs 单日版本

| 对比 | 单日版本 | 多日拼接版本 |
|------|----------|--------------|
| 序列长度 | 288 点 | 288 × N 点 |
| 周期数 | 1 个日周期 | N 个日周期 |
| 频率分辨率 | 1/288 | 1/(288×N) |
| 相位估计 | 噪声大 | 更稳定 |
| 相干性阈值 | 0.3 | 0.5（更严格） |

### 输出文件
| 文件 | 说明 |
|------|------|
| `direction_inference_fft_multiday.csv` | 完整结果 |
| `direction_inference_fft_multiday_summary.txt` | 精度摘要 |
| `direction_inference_fft_multiday.png` | 统计图 |